# 服飾資料庫 × Aiven MySQL 連線測試
密碼執行時用輸入框貼上，**不會寫進檔案**。依序由上往下執行每個儲存格。

In [ ]:
!pip -q install pymysql pandas

## 0. 上傳 Aiven 的 ca.pem 憑證
執行後按「選擇檔案」，選你下載的 `ca.pem`。

In [ ]:
from google.colab import files
up=files.upload()
CA=list(up.keys())[0]; print('已上傳:',CA)

## 1. 連線

In [ ]:
import pymysql, pandas as pd, getpass, urllib.request
HOST=input("Host: ").strip(); PORT=int(input("Port: ").strip())
USER=input("User (例如 avnadmin): ").strip(); PWD=getpass.getpass("Password: ")
def conn(db=None):
    return pymysql.connect(host=HOST,port=PORT,user=USER,password=PWD,database=db,
                           ssl={"ca":CA},autocommit=True,charset="utf8mb4")
c=conn(); print("連線成功，MySQL 版本:", pd.read_sql("SELECT VERSION() v",c).iloc[0,0])

## 2. 從 GitHub 匯入 schema 與資料

In [ ]:
GH_USER=input("GitHub 帳號: ").strip(); REPO=input("儲存庫名稱 (例如 5): ").strip()
RAW=f"https://raw.githubusercontent.com/{GH_USER}/{REPO}/main/sql/"
def run_sql(name):
    txt=urllib.request.urlopen(RAW+name).read().decode("utf-8")
    cur=c.cursor()
    for stmt in [s.strip() for s in txt.split(";\n") if s.strip()]:
        if all(l.strip().startswith("--") or not l.strip() for l in stmt.splitlines()): continue
        cur.execute(stmt)
run_sql("01_schema.sql"); run_sql("02_data.sql"); c.select_db("clothing_store")
print(pd.read_sql("SELECT 'clothing' t,COUNT(*) n FROM clothing UNION ALL SELECT 'orders',COUNT(*) FROM orders UNION ALL SELECT 'order_items',COUNT(*) FROM order_items",c))

## 2. 作業題目查詢（A1–B3）

In [ ]:
Q={
"A1 上衣價格高到低":"SELECT c.product_name,c.price FROM clothing c JOIN categories g USING(category_id) WHERE g.category_name='上衣' ORDER BY c.price DESC",
"B1 分類>2款":"SELECT g.category_name,COUNT(*) n,AVG(c.price) avg_price FROM clothing c JOIN categories g USING(category_id) GROUP BY g.category_id,g.category_name HAVING COUNT(*)>2",
"B3 累計銷售件數":"SELECT c.product_id,c.product_name,COALESCE(SUM(CASE WHEN o.status='已完成' THEN i.quantity END),0) qty FROM clothing c LEFT JOIN order_items i USING(product_id) LEFT JOIN orders o USING(order_id) GROUP BY c.product_id,c.product_name ORDER BY 1"}
for k,v in Q.items(): print("\n##",k); display(pd.read_sql(v,c).head(10))

## 3. 圖片管理驗證（image_url）

In [ ]:
display(pd.read_sql("SELECT COUNT(*) products,COUNT(DISTINCT image_id) distinct_images FROM clothing",c))
display(pd.read_sql("SELECT image_match,COUNT(*) n FROM clothing GROUP BY image_match",c))

## 4. C1 約束測試（三個失敗案例）

In [ ]:
import pymysql
cur=c.cursor()
cur.execute("DROP TABLE IF EXISTS stock_reservation")
cur.execute("""CREATE TABLE stock_reservation(reservation_id INT AUTO_INCREMENT PRIMARY KEY,customer_id VARCHAR(10) NOT NULL,product_id VARCHAR(10) NOT NULL,reserve_date DATE NOT NULL,quantity INT NOT NULL,
CONSTRAINT chk_qty CHECK(quantity BETWEEN 1 AND 4),CONSTRAINT uq_res UNIQUE(customer_id,product_id,reserve_date),
CONSTRAINT fk_c FOREIGN KEY(customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE,
CONSTRAINT fk_p FOREIGN KEY(product_id) REFERENCES clothing(product_id) ON DELETE CASCADE)""")
ins="INSERT INTO stock_reservation(customer_id,product_id,reserve_date,quantity) VALUES(%s,%s,%s,%s)"
for row in [("C001","P0001","2026-03-01",2),("C002","P0002","2026-03-02",1)]: cur.execute(ins,row)
for name,row in [("重複預留",("C001","P0001","2026-03-01",1)),("數量=6",("C003","P0003","2026-03-03",6)),("商品不存在",("C001","P9999","2026-03-04",1))]:
    try: cur.execute(ins,row); print(name,"→ 意外成功！(請檢查該資料庫是否強制 CHECK/FK)")
    except pymysql.MySQLError as e: print(name,"→ 失敗(預期):",e)

## 5. C2：total_sales 函數（MySQL 可用；若權限不足會顯示錯誤，改用下一格）

In [ ]:
cur=c.cursor()
try:
    cur.execute("DROP FUNCTION IF EXISTS total_sales")
    cur.execute("""CREATE FUNCTION total_sales(p_brand_id INT) RETURNS DECIMAL(12,2) READS SQL DATA
BEGIN DECLARE t DECIMAL(12,2);
SELECT COALESCE(SUM(i.quantity*i.unit_price),0) INTO t FROM order_items i JOIN orders o ON o.order_id=i.order_id JOIN clothing c ON c.product_id=i.product_id WHERE c.brand_id=p_brand_id AND o.status='已完成';
RETURN t; END""")
    display(pd.read_sql("SELECT brand_id,brand_name,total_sales(brand_id) sales FROM brands WHERE total_sales(brand_id)>10000",c))
except pymysql.MySQLError as e: print("函數建立失敗:",e)